In [1]:
# Cell 1: 导入依赖 & 配置参数
import re
import psycopg2
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ---------- 数据库连接配置 ----------
DB_CONFIG = {
    "dbname": "Law_app",
    "user": "my_pgsql",
    "password": "123123",
    "host": "localhost",
    "port": 5433,
}

# ---------- 嵌入模型 ----------
MODEL_NAME = "BAAI/bge-large-zh-v1.5"
model = SentenceTransformer(MODEL_NAME)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [2]:
# 测试数据库连接
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT version();")
    version = cur.fetchone()
    print(f"数据库连接成功！PostgreSQL 版本: {version[0]}")
    cur.close()
    conn.close()
except psycopg2.OperationalError as e:
    print(f"数据库连接失败: {e}")
except Exception as e:
    print(f"未知错误: {e}")

数据库连接成功！PostgreSQL 版本: PostgreSQL 15.4 (Debian 15.4-2.pgdg120+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit


In [3]:
def parse_law_from_file(file_path: str):
    """
    从法律文本文件中解析法条。
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict] 每个元素包含 chapter, article_number, content
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ---------- 匹配章/节标题 ----------
    # 支持三种格式: "第X章 XXX"、"X、XXX" 或无章节
    chapter_pattern = re.compile(
        r"^(?:第([一二三四五六七八九十百千零]+)章\s*(.*))"
        r"|^(?:([一二三四五六七八九十百千零]+)、(.+))",
        re.MULTILINE,
    )
    chapter_matches = list(chapter_pattern.finditer(text))

    if not chapter_matches:
        sections = [("", 0, len(text))]
    else:
        sections = []
        for i, m in enumerate(chapter_matches):
            if m.group(1):  # "第X章 XXX" 格式
                chapter_name = f"第{m.group(1)}章 {m.group(2).strip()}"
            else:  # "X、XXX" 格式
                chapter_name = f"{m.group(3)}、{m.group(4).strip()}"
            start = m.start()
            end = chapter_matches[i + 1].start() if i + 1 < len(chapter_matches) else len(text)
            sections.append((chapter_name, start, end))

    # ---------- 匹配条文起始位置 ----------
    article_start_re = re.compile(r"第([一二三四五六七八九十百千零]+)条\s*")

    # ---------- 过滤施行日期条款 ----------
    _DATE_CLAUSE_RE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def is_effective_date_clause(content: str) -> bool:
        c = content.strip()
        return len(c) < 80 and bool(_DATE_CLAUSE_RE.search(c))

    articles = []
    for section_name, sec_start, sec_end in sections:
        section_text = text[sec_start:sec_end]
        # 找到该章节内所有"第X条"的位置
        article_starts = list(article_start_re.finditer(section_text))

        for i, m in enumerate(article_starts):
            article_num = m.group(1)
            content_start = m.end()  # "第X条"之后
            # 内容区间: 当前条文起始 到 下一条文起始(或章节末尾)
            if i + 1 < len(article_starts):
                content_end = article_starts[i + 1].start()
            else:
                content_end = len(section_text)

            raw_content = section_text[content_start:content_end].strip()

            # 以中文句号作为法条内容的自然结束边界
            last_period = raw_content.rfind("。")
            if last_period != -1:
                raw_content = raw_content[:last_period + 1]

            if not raw_content or is_effective_date_clause(raw_content):
                continue

            articles.append(
                {
                    "chapter": section_name or "",
                    "article_number": f"第{article_num}条",
                    "content": raw_content,
                }
            )

    return law_title, articles

In [4]:
# Cell 3: 数据库建表（首次运行执行一次, 表已存在则跳过）
def create_table_if_not_exists():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS marriage_law (
            id SERIAL PRIMARY KEY,
            law_title TEXT NOT NULL,
            chapter TEXT,
            article_number TEXT NOT NULL,
            content TEXT NOT NULL,
            embedding VECTOR(1024),
            UNIQUE(law_title, article_number)
        );
        -- 索引按需创建, 见 create-schema-template.sql
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("表已就绪。")


create_table_if_not_exists()

表已就绪。


In [9]:
# Cell 4: 向量化并插入数据库
def insert_articles(law_title: str, articles: list[dict]):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    sql = """
        INSERT INTO marriage_law (law_title, chapter, article_number, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (law_title, article_number) DO UPDATE
        SET content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            chapter = EXCLUDED.chapter
    """
    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
            ),
        )
    conn.commit()
    cur.close()
    conn.close()
    print(f"成功插入/更新 {len(articles)} 条记录。")

In [5]:
def insert_law_vector(law_title: str, articles: list[dict]):
    """
    将解析后的法条数据插入 law_vector 表。
    Args:
        law_title: 法律名称
        articles: parse_law_from_file 返回的法条列表
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    sql = """
        INSERT INTO law_vector (law_title, chapter, article_number, content, embedding)
        SELECT %s, %s, %s, %s, %s
        WHERE NOT EXISTS (
            SELECT 1 FROM law_vector
            WHERE law_title = %s AND article_number = %s
        )
    """
    inserted = 0
    skipped = 0

    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
                law_title,
                art["article_number"],
            ),
        )
        if cur.rowcount > 0:
            inserted += 1
        else:
            skipped += 1

    conn.commit()
    cur.close()
    conn.close()
    print(f"law_vector: 成功插入 {inserted} 条, 跳过 {skipped} 条(已存在)。")

In [13]:
from pathlib import Path

law_dir = Path(r"E:\\LangChain\\lawApp_LangGraph\\Documents\\LawDocument")
txt_files = sorted(law_dir.glob("*.txt"))

if not txt_files:
    print(f"目录 {law_dir} 下未找到 .txt 文件。")
else:
    print(f"共发现 {len(txt_files)} 个法律文件:\n")
    for fp in txt_files:
        print(f"  - {fp.name}")

    for fp in txt_files:
        print(f"\n处理: {fp.name}")
        law_title, articles = parse_law_from_file(str(fp))
        print(f"  解析到 {len(articles)} 条")
        insert_law_vector(law_title, articles)

共发现 7 个法律文件:

  - 中华人民共和国反家庭暴力法.txt
  - 中华人民共和国妇女权益保障法.txt
  - 中华人民共和国民法典.txt
  - 婚姻登记条例.txt
  - 最高人民法院关于审理涉彩礼纠纷案件.txt
  - 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）.txt
  - 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）.txt

处理: 中华人民共和国反家庭暴力法.txt
  解析到 38 条
law_vector: 成功插入 37 条, 跳过 1 条(已存在)。

处理: 中华人民共和国妇女权益保障法.txt
  解析到 89 条
law_vector: 成功插入 85 条, 跳过 4 条(已存在)。

处理: 中华人民共和国民法典.txt
  解析到 1320 条
law_vector: 成功插入 1258 条, 跳过 62 条(已存在)。

处理: 婚姻登记条例.txt
  解析到 32 条
law_vector: 成功插入 29 条, 跳过 3 条(已存在)。

处理: 最高人民法院关于审理涉彩礼纠纷案件.txt
  解析到 7 条
law_vector: 成功插入 7 条, 跳过 0 条(已存在)。

处理: 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）.txt
  解析到 131 条
law_vector: 成功插入 106 条, 跳过 25 条(已存在)。

处理: 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）.txt
  解析到 40 条
law_vector: 成功插入 36 条, 跳过 4 条(已存在)。


In [11]:
# Cell 5: 假数据测试（使用样例文本）
sample_text = """第六章　附　　则

第二十五条　中华人民共和国驻外使（领）馆可以依照本条例的有关规定，为男女双方均居住于驻在国的中国公民办理婚姻登记。
第二十六条　男女双方均非内地居民的中国公民在内地办理婚姻登记的具体办法，由国务院民政部门另行制定。
第二十七条　本条例规定的婚姻登记证由国务院民政部门规定式样并监制。
第二十八条　本条例自2025年5月10日起施行。"""

# 将样例文本写入临时文件
sample_file = "sample_law.txt"
with open(sample_file, "w", encoding="utf-8") as f:
    f.write(sample_text)

# 解析
law_title, articles = parse_law_from_file(sample_file)
print(f"法律名称：{law_title}")
for a in articles:
    print(f"{a['article_number']} [{a['chapter']}] {a['content'][:30]}...")

# 插入数据库（第二十八条会被自动过滤）
insert_articles(law_title, articles)

# 验证：查询一条记录
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute(
    "SELECT law_title, article_number, content FROM marriage_law WHERE law_title=%s LIMIT 1;",
    (law_title,),
)
row = cur.fetchone()
print(f"数据库验证：{row}")
cur.close()
conn.close()

法律名称：sample_law
第二十五条 [第六章 附　　则] 中华人民共和国驻外使（领）馆可以依照本条例的有关规定，为男女...
第二十六条 [第六章 附　　则] 男女双方均非内地居民的中国公民在内地办理婚姻登记的具体办法，...
第二十七条 [第六章 附　　则] 本条例规定的婚姻登记证由国务院民政部门规定式样并监制。...
成功插入/更新 3 条记录。
数据库验证：('sample_law', '第二十七条', '本条例规定的婚姻登记证由国务院民政部门规定式样并监制。')


In [7]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="127.0.0.1",
        port=5433,
        database="Law_app",
        user="my_pgsql",
        password="123123",
        options="-c client_encoding=UTF8",
    )
    cur = conn.cursor()
    # 查询数据库版本
    cur.execute("SELECT version();")
    version = cur.fetchone()
    print("连接成功！")
    print("PostgreSQL 版本:", version[0])

    # 查询 law_vector 表是否存在
    cur.execute(
        "SELECT EXISTS (SELECT FROM information_schema.tables WHERE table_name = 'law_vector');"
    )
    table_exists = cur.fetchone()[0]
    print("law_vector 表存在:", table_exists)

    cur.close()
    conn.close()
except psycopg2.OperationalError as e:
    print("连接失败:", e)


连接成功！
PostgreSQL 版本: PostgreSQL 15.4 (Debian 15.4-2.pgdg120+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit
law_vector 表存在: True


In [ ]:
def parse_law_from_file_Version2(file_path: str):
    """
    从法律文本文件中解析法条。
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict] 每个元素包含 chapter, article_number, content
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ---------- 匹配章/节标题 ----------
    # 支持三种格式: "第X章 XXX"、"X、XXX" 或无章节
    chapter_pattern = re.compile(
        r"^(?:第([一二三四五六七八九十百千零]+)章\s*(.*))"
        r"|^(?:([一二三四五六七八九十百千零]+)、(.+))",
        re.MULTILINE,
    )
    chapter_matches = list(chapter_pattern.finditer(text))

    if not chapter_matches:
        sections = [("", 0, len(text))]
    else:
        sections = []
        for i, m in enumerate(chapter_matches):
            if m.group(1):  # "第X章 XXX" 格式
                chapter_name = f"第{m.group(1)}章 {m.group(2).strip()}"
            else:  # "X、XXX" 格式
                chapter_name = f"{m.group(3)}、{m.group(4).strip()}"
            start = m.start()
            end = (
                chapter_matches[i + 1].start()
                if i + 1 < len(chapter_matches)
                else len(text)
            )
            sections.append((chapter_name, start, end))

    # ---------- 匹配条文起始位置 ----------
    article_start_re = re.compile(r"第([一二三四五六七八九十百千零]+)条\s*")

    # ---------- 过滤施行日期条款 ----------
    _DATE_CLAUSE_RE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def is_effective_date_clause(content: str) -> bool:
        c = content.strip()
        return len(c) < 80 and bool(_DATE_CLAUSE_RE.search(c))

    articles = []
    for section_name, sec_start, sec_end in sections:
        section_text = text[sec_start:sec_end]
        # 找到该章节内所有"第X条"的位置
        article_starts = list(article_start_re.finditer(section_text))

        for i, m in enumerate(article_starts):
            article_num = m.group(1)
            content_start = m.end()  # "第X条"之后
            # 内容区间: 当前条文起始 到 下一条文起始(或章节末尾)
            if i + 1 < len(article_starts):
                content_end = article_starts[i + 1].start()
            else:
                content_end = len(section_text)

            raw_content = section_text[content_start:content_end].strip()

            # 以中文句号作为法条内容的自然结束边界
            last_period = raw_content.rfind("。")
            if last_period != -1:
                raw_content = raw_content[: last_period + 1]

            if not raw_content or is_effective_date_clause(raw_content):
                continue

            articles.append(
                {
                    "chapter": section_name or "",
                    "article_number": f"第{article_num}条",
                    "content": raw_content,
                }
            )

    return law_title, articles


: 